# 01 — Classify a complaint

Vertical slice: one complaint in, one evidence record out. This proves the
classification path end to end before anything is batched (see
data/synthetic/generate.py for the 200-record batch, run later once this is
validated).

**Run this after:**
- The workbench is running with the taxonomy ConfigMap mounted
  (manifests/workbench/workbench.yaml)
- `ansible-playbook ansible/site.yml` has completed, including
  seed_evidence_bucket

## Environment variables required

| Variable | Description | Default |
|---|---|---|
| `LLAMA_STACK_URL` | Llama Stack service | `http://lsd-complaint-intelligence-service:8321` |
| `GUARDRAILS_URL` | Guardrails orchestrator, HTTPS, self-signed | `https://guardrails-orchestrator-service:8032` |
| `MINIO_ENDPOINT` | MinIO service | `minio.complaint-intelligence.svc.cluster.local:9000` |
| `MINIO_ACCESS_KEY` | MinIO access key | none, required |
| `MINIO_SECRET_KEY` | MinIO secret key | none, required |
| `TAXONOMY_PATH` | Mounted ConfigMap path | `/opt/app-root/taxonomy/taxonomy.yaml` |

One open item this notebook will resolve live: the retrieval/vector query
call against Llama Stack is not yet validated on RHOAI 3.4.2 (see Cell 7).
Everything else below uses calls already confirmed working in
DEPLOYMENT-LOG-2026-07-15-rhoai34.md.

## Cell 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install --quiet requests==2.32.3 pyyaml==6.0.2 minio==7.2.7
print("Dependencies installed.")

## Cell 2 — Configuration

In [ ]:
import os

LLAMA_STACK_URL  = os.environ.get("LLAMA_STACK_URL", "http://lsd-complaint-intelligence-service:8321")
GUARDRAILS_URL   = os.environ.get("GUARDRAILS_URL", "https://guardrails-orchestrator-service:8032")
TAXONOMY_PATH    = os.environ.get("TAXONOMY_PATH", "/opt/app-root/taxonomy/taxonomy.yaml")

MINIO_ENDPOINT   = os.environ.get("MINIO_ENDPOINT", "minio.complaint-intelligence.svc.cluster.local:9000")
MINIO_ACCESS_KEY = os.environ["MINIO_ACCESS_KEY"]   # fail fast if not set
MINIO_SECRET_KEY = os.environ["MINIO_SECRET_KEY"]   # fail fast if not set
MINIO_BUCKET     = "uc02"

COMPLAINTS_KEY   = "complaints/incoming/records.jsonl"
EVIDENCE_PREFIX  = "evidence/classifications"

print("Configuration loaded.")

## Cell 3 — Load taxonomy from the mounted ConfigMap

In [ ]:
import yaml
from pathlib import Path

taxonomy_file = Path(TAXONOMY_PATH)
if not taxonomy_file.exists():
    raise RuntimeError(
        f"Taxonomy not found at {TAXONOMY_PATH}. Confirm the taxonomy "
        f"ConfigMap volume is mounted (manifests/workbench/workbench.yaml)."
    )

taxonomy = yaml.safe_load(taxonomy_file.read_text())
TAXONOMY_VERSION = taxonomy["version"]
CONFIDENCE_THRESHOLD = taxonomy["classification"]["confidence_threshold"]

print(f"Taxonomy version {TAXONOMY_VERSION}: "
      f"{len(taxonomy['themes'])} themes, {len(taxonomy['root_causes'])} root causes")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

## Cell 4 — Discover the model at runtime

In [ ]:
import requests

resp = requests.get(f"{LLAMA_STACK_URL}/v1/models")
resp.raise_for_status()
models = resp.json().get("data", resp.json())

llm_candidates = [m for m in models if "granite" in m.get("id", "").lower()]
if not llm_candidates:
    raise RuntimeError(f"No Granite model found in /v1/models. Got: {models}")

MODEL_ID = llm_candidates[0]["id"]
print(f"Using model: {MODEL_ID}")

## Cell 5 — Pull one complaint from MinIO

In [ ]:
from minio import Minio
import json

minio_client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY,
                      secret_key=MINIO_SECRET_KEY, secure=False)

if not minio_client.bucket_exists(MINIO_BUCKET):
    raise RuntimeError(f"Bucket '{MINIO_BUCKET}' not found. Run ansible/site.yml.")

response = minio_client.get_object(MINIO_BUCKET, COMPLAINTS_KEY)
first_line = response.read().decode("utf-8").splitlines()[0]
response.close()
response.release_conn()

complaint = json.loads(first_line)
print(f"Loaded complaint {complaint['complaint_id']}")
print(complaint["text"][:300])

## Cell 6 — Guardrails input check

In [ ]:
detection_payload = {
    "detectors": {"regex": {"regex": ["email", "credit-card"]}},
    "content": complaint["text"]
}

# TLS is self-signed in-cluster (validated finding, DEPLOYMENT-LOG-2026-07-15).
detection_resp = requests.post(
    f"{GUARDRAILS_URL}/api/v2/text/detection/content",
    json=detection_payload,
    verify=False
)
detection_resp.raise_for_status()
detections = detection_resp.json()

pii_detected = len(detections) > 0
pii_redactions = len(detections)

# Note: only email/credit-card regex detectors are configured
# (manifests/guardrails/orchestrator-config.yaml). No injection detector is
# currently registered, so injection_blocked is not yet a live check, it is
# recorded honestly as not evaluated rather than assumed false.
injection_blocked = None

print(f"PII detected: {pii_detected} ({pii_redactions} matches)")
print(f"Injection check: not configured on this stack")

## Cell 7 — Retrieval discovery (live validation needed)

In [ ]:
resp = requests.get(f"{LLAMA_STACK_URL}/v1/inspect/routes")
resp.raise_for_status()
routes = resp.json()

vector_routes = [r for r in routes.get("data", routes) if "vector" in str(r).lower()]

if not vector_routes:
    raise RuntimeError(
        "No vector/retrieval routes found via /v1/inspect/routes. Confirm "
        "ENABLE_INLINE_MILVUS is set on the LlamaStackDistribution CR "
        "(ADR-0008) before continuing."
    )

print("Candidate vector/retrieval routes:")
for r in vector_routes:
    print(" ", r)

# STOP HERE on first run. Inspect the printed routes, confirm the query
# call and response shape against this cluster, then replace this cell
# with the validated retrieval call before continuing. Once confirmed,
# capture the finding as an addition to ADR-0008.
retrieved_context = ""  # placeholder until the above is validated

## Cell 8 — Build the classification prompt

In [ ]:
theme_block = "\n".join(
    f"- {t['id']}: {t['name']} — {t['definition'].strip()}"
    for t in taxonomy["themes"]
)
root_cause_block = "\n".join(
    f"- {r['id']}: {r['name']} — {r['definition'].strip()}"
    for r in taxonomy["root_causes"]
)

prompt = f"""You are classifying a bank complaint against a fixed taxonomy.

Themes:
{theme_block}

Root causes:
{root_cause_block}

{"Related context:\n" + retrieved_context if retrieved_context else ""}

Complaint:
\"\"\"{complaint['text']}\"\"\"

Respond with JSON only, no other text, in this exact shape:
{{
  "theme_id": "THM-XX",
  "root_cause_id": "RC-XX",
  "confidence": 0.0,
  "citation_text": "the exact sentence from the complaint that supports this classification",
  "candidate_themes": [{{"theme_id": "THM-XX", "score": 0.0}}, ...]
}}

candidate_themes must include every theme you considered plausible, ranked
by score, even if only one. citation_text must be copied verbatim from the
complaint text above."""

print(prompt[:500], "...")

## Cell 9 — Call the model and parse the response

In [ ]:
completion_resp = requests.post(
    f"{LLAMA_STACK_URL}/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 500,
        "temperature": 0.0
    }
)
completion_resp.raise_for_status()
raw = completion_resp.json()["choices"][0]["message"]["content"]

try:
    classification = json.loads(raw)
except json.JSONDecodeError:
    raise RuntimeError(f"Model did not return valid JSON:\n{raw}")

print(json.dumps(classification, indent=2))

## Cell 10 — Locate the citation and apply the review threshold

In [ ]:
citation_text = classification["citation_text"]
start = complaint["text"].find(citation_text)

if start == -1:
    # Model paraphrased instead of quoting verbatim. Flag rather than guess
    # an offset, since a wrong citation is worse than a missing one.
    citation = {"start": None, "end": None, "text": citation_text}
    citation_verified = False
else:
    citation = {"start": start, "end": start + len(citation_text), "text": citation_text}
    citation_verified = True

confidence = classification["confidence"]
routed_to_review = confidence < CONFIDENCE_THRESHOLD

if routed_to_review:
    sorted_candidates = sorted(classification["candidate_themes"],
                                key=lambda c: c["score"], reverse=True)
    review_reason = (
        f"confidence {confidence:.2f} below threshold {CONFIDENCE_THRESHOLD}"
    )
    candidate_theme_ids = [c["theme_id"] for c in sorted_candidates]
else:
    review_reason = None
    candidate_theme_ids = []

print(f"Routed to review: {routed_to_review}")
print(f"Citation verified against source text: {citation_verified}")

## Cell 11 — Assemble and write the evidence record

In [ ]:
from datetime import datetime, timezone
import uuid
import io

evidence_record = {
    "complaint_id": complaint["complaint_id"],
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "theme_id": classification["theme_id"],
    "root_cause_id": classification["root_cause_id"],
    "confidence": confidence,
    "citation": citation,
    "routed_to_review": routed_to_review,
    "review_reason": review_reason,
    "candidate_themes": candidate_theme_ids,
    "pii_detected": pii_detected,
    "pii_redactions": pii_redactions,
    "injection_blocked": injection_blocked,
    "guardrail_policy_id": "regex",
    "prompt_version": "0.1.0",
    "model_version": MODEL_ID,
    "taxonomy_version": TAXONOMY_VERSION,
    "trace_id": completion_resp.json().get("id", str(uuid.uuid4()))
}

record_bytes = json.dumps(evidence_record).encode("utf-8")
record_key = f"{EVIDENCE_PREFIX}/{complaint['complaint_id']}.json"
minio_client.put_object(
    MINIO_BUCKET, record_key,
    data=io.BytesIO(record_bytes),
    length=len(record_bytes),
    content_type="application/json"
)

print(f"Evidence record written to s3://{MINIO_BUCKET}/{record_key}")
print(json.dumps(evidence_record, indent=2))